<a href="https://colab.research.google.com/github/kilianodonell-cmd/Q3_Durban/blob/copilot/make-html-accessible/Field_Map_Deep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount Google Drive only when running in Google Colab
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive', force_remount=True)
    print('Colab environment detected. Google Drive mounted.')
else:
    print('Local environment detected. Skipping Google Drive mount.')

print('Setup complete.')

Mounted at /content/drive
Setup complete.


In [2]:
import os, json, numpy as np
import geopandas as gpd
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
import folium
from folium import LayerControl
from PIL import Image
import base64, io
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
# ============================================================
# CELL 1 - Setup
# Field Map - Housing Suitability
# Mzinyati Stream Catchment, eThekwini Municipality
# ============================================================

# Load config (created by MCA notebook)
# Priority order:
# 1) FIELD_MAP_CONFIG_PATH environment variable
# 2) Local repo outputs
# 3) Colab default path
candidate_config_paths = []

env_config = os.environ.get('FIELD_MAP_CONFIG_PATH', '').strip()
if env_config:
    candidate_config_paths.append(env_config)

# Local workspace candidate path
candidate_config_paths.append(os.path.join(os.getcwd(), 'outputs', 'field_map_config.json'))

# Colab default candidate path
candidate_config_paths.append('/content/drive/MyDrive/Durban/outputs/field_map_config.json')

CONFIG_PATH = None
for path in candidate_config_paths:
    if os.path.exists(path):
        CONFIG_PATH = path
        break

if CONFIG_PATH is None:
    raise FileNotFoundError(
        'Could not find field_map_config.json. Set FIELD_MAP_CONFIG_PATH or generate outputs from Durban_MCA.ipynb.'
    )

with open(CONFIG_PATH) as f:
    config = json.load(f)

OUTPUT_ROOT = config['OUTPUT_ROOT']
TARGET_CRS = config['TARGET_CRS']
SCENARIOS = config['SCENARIOS']
SUIT_COLORS = config['SUIT_COLORS']
SUIT_LABELS = config['SUIT_LABELS']
AOI_FIELD = config['AOI_FIELD']
AOI_VALUE = config['AOI_VALUE']
CATCHMENTS_PATH = config['CATCHMENTS_PATH']

RASTERS_DIR = os.path.join(OUTPUT_ROOT, 'rasters')
BUILDINGS_PATH = os.path.join(OUTPUT_ROOT, 'at_risk_buildings.gpkg')
CLIPPED_DIR = os.path.join(OUTPUT_ROOT, 'clipped')

print('=' * 60)
print('FIELD MAP SETUP')
print('=' * 60)
print(f'  Config path: {CONFIG_PATH}')
print(f'  Output root: {OUTPUT_ROOT}')
print(f'  Scenarios:   {list(SCENARIOS.keys())}')

# Check that required files exist
print('\n  Checking files...')
missing = []

for scenario in SCENARIOS:
    raster_path = os.path.join(RASTERS_DIR, f'suitability_{scenario}.tif')
    if os.path.exists(raster_path):
        print(f'    ok suitability_{scenario}.tif')
    else:
        print(f'    missing suitability_{scenario}.tif')
        missing.append(raster_path)

constraint_path = os.path.join(RASTERS_DIR, 'constraint_mask.tif')
if os.path.exists(constraint_path):
    print('    ok constraint_mask.tif')
else:
    print('    missing constraint_mask.tif')
    missing.append(constraint_path)

if os.path.exists(BUILDINGS_PATH):
    print('    ok at_risk_buildings.gpkg')
else:
    print('    missing at_risk_buildings.gpkg')
    missing.append(BUILDINGS_PATH)

if missing:
    print(f'\n  Warning: {len(missing)} files missing. Run Durban_MCA.ipynb first.')
else:
    print('\n  All files ready. Proceed to the map cell.')

print('=' * 60)

FIELD MAP SETUP
  Output root: /content/drive/MyDrive/Durban/outputs
  Scenarios:   ['hazard_focused', 'balanced', 'infrastructure_focused']

  Checking files...
    ✓ suitability_hazard_focused.tif
    ✓ suitability_balanced.tif
    ✓ suitability_infrastructure_focused.tif
    ✓ constraint_mask.tif
    ✓ at_risk_buildings.gpkg

  ✓ All files ready. Proceed to CELL 2.


In [ ]:
# ============================================================
# CELL 2 - Precompute Building Factor Scores (from MCA rasters)
# Option 1: Keep MCA unchanged, sample scored_*.tif here once
# ============================================================

import glob
import pandas as pd

AT_RISK_PATH = os.path.join(OUTPUT_ROOT, 'at_risk_buildings.gpkg')
FACTOR_RASTER_PATTERN = os.path.join(RASTERS_DIR, 'scored_*.tif')

if not os.path.exists(AT_RISK_PATH):
    raise FileNotFoundError(f'Missing buildings layer: {AT_RISK_PATH}')

factor_rasters = sorted(glob.glob(FACTOR_RASTER_PATTERN))
if not factor_rasters:
    raise FileNotFoundError(f'No factor rasters found matching: {FACTOR_RASTER_PATTERN}')

prepared_buildings = gpd.read_file(AT_RISK_PATH)

if 'in_constraint' not in prepared_buildings.columns:
    raise ValueError("Expected 'in_constraint' column in at_risk_buildings.gpkg")

# Identify a stable ID column for popups.
def pick_id_column(df):
    id_candidates = [
        'building_id', 'BUILDING_ID', 'id', 'ID', 'fid', 'FID',
        'objectid', 'OBJECTID', 'osm_id', 'OSM_ID'
    ]
    for c in id_candidates:
        if c in df.columns:
            return c
    return None

popup_id_col = pick_id_column(prepared_buildings)
if popup_id_col is None:
    popup_id_col = 'building_index'
    prepared_buildings[popup_id_col] = prepared_buildings.index.astype(str)

# Sample each scored factor raster at building centroids.
for raster_path in factor_rasters:
    factor_name = os.path.splitext(os.path.basename(raster_path))[0].replace('scored_', '', 1)
    factor_col = f'factor_score_{factor_name}'

    with rasterio.open(raster_path) as src:
        src_crs = src.crs
        nodata = src.nodata

        gdf_src = prepared_buildings.to_crs(src_crs)
        centroids = gdf_src.geometry.centroid
        coords = [(geom.x, geom.y) for geom in centroids]

        sampled = [val[0] for val in src.sample(coords)]
        sampled = pd.to_numeric(pd.Series(sampled), errors='coerce')

        if nodata is not None:
            sampled = sampled.where(sampled != nodata)

        # Keep only valid suitability class values.
        sampled = sampled.where(sampled.isin([1, 2, 3, 4, 5]))
        prepared_buildings[factor_col] = sampled.values

factor_cols = [c for c in prepared_buildings.columns if c.startswith('factor_score_')]
if not factor_cols:
    raise ValueError('No factor_score_* columns were created from scored rasters.')

# Build top-3 worst factor text (5 = worst) for each building.
def nice_factor_name(col_name):
    return col_name.replace('factor_score_', '').replace('_', ' ').strip().title()

def compute_top3_worst_text(row, cols):
    scored = []
    for c in cols:
        v = row.get(c)
        if pd.isna(v):
            continue
        fv = float(v)
        if fv < 1 or fv > 5:
            continue
        scored.append((fv, nice_factor_name(c)))

    if not scored:
        return 'Factor scores unavailable'

    scored.sort(key=lambda x: (-x[0], x[1]))
    top = scored[:3]
    return ', '.join([f'{name} ({int(val)})' for val, name in top])

# Keep scenario-specific popup columns for compatibility with scenario layers.
for scenario in SCENARIOS.keys():
    prepared_buildings[f'top3_worst_{scenario}'] = prepared_buildings.apply(
        lambda row: compute_top3_worst_text(row, factor_cols),
        axis=1,
    )

print('=' * 60)
print('PRECOMPUTE COMPLETE')
print('=' * 60)
print(f'Factor rasters found: {len(factor_rasters)}')
print(f'Factor columns created: {len(factor_cols)}')
print(f'Popup ID column: {popup_id_col}')
print('Prepared data is ready. Run next cell to create the map.')

In [ ]:
# ============================================================
# FIELD MAP DEEP - INTERACTIVE POC MAP
# - Uses precomputed factor scores from previous cell
# - Dynamic scenario layers from MCA config (no hardcoding)
# - Scenario suitability masks visible as raster overlays
# - Building GeoJSON embedded once; JS handles scenario switch
# ============================================================

import os
import json as _json
import numpy as np
import folium
from folium import plugins
import geopandas as gpd
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape, box
from branca.element import Element
from IPython.display import IFrame, display

print('=' * 60)
print('FIELD MAP DEEP POC')
print('=' * 60)

# ------------------------------------------------------------
# LOAD PREPARED DATA
# ------------------------------------------------------------
if 'prepared_buildings' not in globals():
    raise RuntimeError('Please run Cell 2 (precompute factors) before running this map cell.')

if 'popup_id_col' not in globals():
    raise RuntimeError('Missing popup_id_col. Please run Cell 2 first.')

buildings = prepared_buildings.copy()
scenario_keys = list(SCENARIOS.keys())

AT_RISK_PATH = os.path.join(OUTPUT_ROOT, 'at_risk_buildings.gpkg')
CONSTRAINT_RASTER = os.path.join(RASTERS_DIR, 'constraint_mask.tif')

if not os.path.exists(CONSTRAINT_RASTER):
    raise FileNotFoundError(f'Missing constraint raster: {CONSTRAINT_RASTER}')

# Validate that scenario columns exist in the buildings layer.
missing_cols = []
for scenario in scenario_keys:
    dcol = f'display_score_{scenario}'
    if dcol not in buildings.columns:
        missing_cols.append(dcol)

if missing_cols:
    missing_msg = '\n'.join([f'  - {c}' for c in missing_cols])
    raise ValueError(
        'Scenario display score columns missing from prepared buildings:\n' + missing_msg
    )

buildings = buildings.to_crs(4326)
buildings['geometry'] = buildings.geometry.simplify(0.00001, preserve_topology=True)

# ---- TRIM COLUMNS (keeps GeoJSON small; drops factor_score_* etc.) ----
_geom_col = buildings.geometry.name
_keep = (
    [popup_id_col, 'in_constraint']
    + [f'display_score_{s}' for s in scenario_keys if f'display_score_{s}' in buildings.columns]
    + [f'top3_worst_{s}'    for s in scenario_keys if f'top3_worst_{s}'    in buildings.columns]
)
_keep = list(dict.fromkeys(c for c in _keep if c in buildings.columns))
buildings = buildings[_keep + [_geom_col]]

constraint_buildings     = buildings[buildings['in_constraint'] == True].copy()
non_constraint_buildings = buildings[buildings['in_constraint'] != True].copy()

# Load AOI boundary
catchments = gpd.read_file(CATCHMENTS_PATH)
aoi = catchments[catchments[AOI_FIELD] == AOI_VALUE].copy().to_crs(4326)
aoi['geometry'] = aoi.geometry.simplify(0.00001, preserve_topology=True)

# ------------------------------------------------------------
# BUILD CONSTRAINT AREA FROM RASTER
# ------------------------------------------------------------
constraint_mask_gdf = gpd.GeoDataFrame(geometry=[], crs='EPSG:4326')

with rasterio.open(CONSTRAINT_RASTER) as src:
    mask_data = src.read(1)
    transform = src.transform
    crs = src.crs
    nodata = src.nodata

    print('Constraint raster unique values:', np.unique(mask_data))

    valid_mask = (mask_data != nodata) if nodata is not None else np.ones(mask_data.shape, dtype=bool)

    polygons = []
    for geom, value in shapes(mask_data, mask=valid_mask, transform=transform):
        if value == 0:
            polygons.append(shape(geom))

    if polygons:
        constraint_mask_gdf = gpd.GeoDataFrame(geometry=polygons, crs=crs).to_crs(4326)
        constraint_mask_gdf['geometry'] = constraint_mask_gdf.geometry.simplify(0.00001, preserve_topology=True)

# ------------------------------------------------------------
# MAP BASE
# ------------------------------------------------------------
center = aoi.union_all().centroid

print(f'AOI: {AOI_VALUE}')
print(f'Total buildings: {len(buildings):,}')
print(f'In-constraint buildings: {len(constraint_buildings):,}')
print(f'Non-constraint buildings: {len(non_constraint_buildings):,}')
print(f'Constraint polygons: {len(constraint_mask_gdf):,}')

m = folium.Map(
    location=[center.y, center.x],
    zoom_start=16,
    max_zoom=22,
    tiles='https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',
    attr='&copy; <a href="https://www.openstreetmap.org/copyright">OSM</a> &copy; CARTO',
    control_scale=True,
)

# AOI boundary
folium.GeoJson(
    aoi,
    name='AOI boundary',
    style_function=lambda feat: {
        'fillColor': 'none',
        'color': '#111111',
        'weight': 2,
        'dashArray': '5, 5',
    },
).add_to(m)

# ------------------------------------------------------------
# SCENARIO MASK RASTER OVERLAYS (downsampled to reduce file size)
# ------------------------------------------------------------
_MAX_RASTER_DIM = 768  # cap longest side (pixels) before base64 encoding

def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

scenario_mask_js_vars = {}

for idx, scenario in enumerate(scenario_keys):
    scenario_raster = os.path.join(RASTERS_DIR, f'suitability_{scenario}.tif')
    if not os.path.exists(scenario_raster):
        print(f'Skipping scenario mask (missing raster): {scenario}')
        continue

    with rasterio.open(scenario_raster) as src:
        arr = src.read(1)
        nodata = src.nodata
        if nodata is not None:
            arr = np.where(arr == nodata, 0, arr)

        rgba = np.zeros((arr.shape[0], arr.shape[1], 4), dtype=np.uint8)
        for class_value in [1, 2, 3, 4, 5]:
            class_mask = arr == class_value
            if class_mask.any():
                r, g, b = hex_to_rgb(SUIT_COLORS[class_value - 1])
                rgba[class_mask, 0] = r
                rgba[class_mask, 1] = g
                rgba[class_mask, 2] = b
                rgba[class_mask, 3] = 125

        # Downsample to reduce embedded base64 image size
        _h, _w = rgba.shape[:2]
        if max(_h, _w) > _MAX_RASTER_DIM:
            _scale = _MAX_RASTER_DIM / max(_h, _w)
            _pil = Image.fromarray(rgba).resize(
                (max(1, int(_w * _scale)), max(1, int(_h * _scale))),
                Image.LANCZOS,
            )
            rgba = np.array(_pil)

        raster_bounds_geom = gpd.GeoSeries(
            [box(src.bounds.left, src.bounds.bottom, src.bounds.right, src.bounds.top)],
            crs=src.crs,
        ).to_crs(4326)
        minx, miny, maxx, maxy = raster_bounds_geom.total_bounds

        ia = folium.raster_layers.ImageOverlay(
            image=rgba,
            bounds=[[miny, minx], [maxy, maxx]],
            name=f'Scenario mask: {scenario}',
            opacity=0.65,
            interactive=False,
            show=True if idx == 0 else False,
            cross_origin=False,
            zindex=1,
        )
        ia.add_to(m)
        scenario_mask_js_vars[scenario] = ia.get_name()

# Constraint area layer
if len(constraint_mask_gdf) > 0:
    folium.GeoJson(
        constraint_mask_gdf,
        name='Filter: Constraint area',
        style_function=lambda feat: {
            'fillColor': '#dc2626',
            'color': '#991b1b',
            'weight': 0.7,
            'fillOpacity': 0.28,
        },
        tooltip='Constraint area',
        show=True,
    ).add_to(m)

# ------------------------------------------------------------
# BUILDING LAYERS — single Leaflet layer per group, JS scenario
# switching avoids embedding GeoJSON N times for N scenarios
# ------------------------------------------------------------
_map_var    = m.get_name()
_popup_id   = popup_id_col
_first_sc   = scenario_keys[0]
_nc_geojson = non_constraint_buildings.to_json()
_c_geojson  = (
    constraint_buildings.to_json()
    if len(constraint_buildings) > 0
    else '{"type":"FeatureCollection","features":[]}'
)

_colors_js    = _json.dumps(SUIT_COLORS)
_scenarios_js = _json.dumps(scenario_keys)
_masks_js     = _json.dumps(scenario_mask_js_vars)

buildings_js_src = (
    '<script>\n'
    '(function () {\n'
    '  var _ncData    = ' + _nc_geojson + ';\n'
    '  var _cData     = ' + _c_geojson  + ';\n'
    '  var _COLORS    = ' + _colors_js  + ';\n'
    '  var _scenarios = ' + _scenarios_js + ';\n'
    '  var _maskVars  = ' + _masks_js   + ';\n'
    '  var _active    = "' + _first_sc  + '";\n'
    '  var _pid       = "' + _popup_id  + '";\n'
    '\n'
    '  function getColor(score) {\n'
    '    var s = parseInt(score);\n'
    '    return (s >= 1 && s <= 5) ? _COLORS[s - 1] : "#9e9e9e";\n'
    '  }\n'
    '\n'
    '  function ncStyle(f) {\n'
    '    var score = f.properties["display_score_" + _active];\n'
    '    return {fillColor: getColor(score), color: "#2d2d2d", weight: 0.2, fillOpacity: 0.85};\n'
    '  }\n'
    '\n'
    '  function makePopup(p) {\n'
    '    var sc = _active;\n'
    '    return "<b>Building ID:</b> " + (p[_pid] || "") +\n'
    '           "<br><b>Display score:</b> " + (p["display_score_" + sc] || "") +\n'
    '           "<br><b>Top 3 worst factors:</b> " + (p["top3_worst_" + sc] || "");\n'
    '  }\n'
    '\n'
    '  function attachFeature(f, layer) {\n'
    '    layer.bindTooltip(String(f.properties["display_score_" + _active] || ""), {sticky: false});\n'
    '    layer.on("click", function () {\n'
    '      layer.bindPopup(makePopup(f.properties)).openPopup();\n'
    '    });\n'
    '  }\n'
    '\n'
    '  function waitForMap() {\n'
    '    var mapObj = window["' + _map_var + '"];\n'
    '    if (!mapObj) { setTimeout(waitForMap, 100); return; }\n'
    '\n'
    '    var ncLayer = L.geoJson(_ncData, {style: ncStyle, onEachFeature: attachFeature}).addTo(mapObj);\n'
    '    var cLayer  = L.geoJson(_cData, {\n'
    '      style: function () {\n'
    '        return {fillColor: "#7f1a1a", color: "#3f0a0a", weight: 0.25, fillOpacity: 0.9};\n'
    '      },\n'
    '      onEachFeature: function (f, layer) {\n'
    '        layer.on("click", function () {\n'
    '          layer.bindPopup(makePopup(f.properties)).openPopup();\n'
    '        });\n'
    '      }\n'
    '    }).addTo(mapObj);\n'
    '\n'
    '    window._ncLayer = ncLayer;\n'
    '    window._cLayer  = cLayer;\n'
    '\n'
    '    window.switchScenario = function (scenario) {\n'
    '      _active = scenario;\n'
    '      ncLayer.setStyle(function (f) {\n'
    '        var score = f.properties["display_score_" + scenario];\n'
    '        return {fillColor: getColor(score), color: "#2d2d2d", weight: 0.2, fillOpacity: 0.85};\n'
    '      });\n'
    '      ncLayer.eachLayer(function (l) {\n'
    '        l.unbindTooltip();\n'
    '        l.bindTooltip(String(l.feature.properties["display_score_" + scenario] || ""), {sticky: false});\n'
    '      });\n'
    '      // Show matching scenario raster, hide others\n'
    '      _scenarios.forEach(function (s) {\n'
    '        var lyr = window[_maskVars[s]];\n'
    '        if (!lyr) return;\n'
    '        if (s === scenario) { if (!mapObj.hasLayer(lyr)) mapObj.addLayer(lyr); }\n'
    '        else mapObj.removeLayer(lyr);\n'
    '      });\n'
    '      // Highlight active tab\n'
    '      document.querySelectorAll(".sc-tab").forEach(function (btn) {\n'
    '        var active = btn.dataset.scenario === scenario;\n'
    '        btn.style.background  = active ? "#374151" : "#e5e7eb";\n'
    '        btn.style.color       = active ? "#ffffff"  : "#374151";\n'
    '        btn.style.fontWeight  = active ? "700"      : "400";\n'
    '      });\n'
    '    };\n'
    '  }\n'
    '\n'
    '  waitForMap();\n'
    '}})();\n'
'</script>'
)
m.get_root().html.add_child(Element(buildings_js_src))

# Scenario switcher tab bar (floating, top-centre)
_tabs = ''.join(
    '<button class="sc-tab" data-scenario="' + s + '" onclick="switchScenario(\'' + s + '\')" '
    'style="padding:4px 12px;border:1px solid #d1d5db;border-radius:4px;cursor:pointer;font-size:11px;'
    'background:' + ('#374151' if i == 0 else '#e5e7eb') + ';'
    'color:' + ('#ffffff' if i == 0 else '#374151') + ';'
    'font-weight:' + ('700' if i == 0 else '400') + ';">' + s + '</button>'
    for i, s in enumerate(scenario_keys)
)
scenario_ctrl_html = (
    '<div style="position:fixed;top:12px;left:50%;transform:translateX(-50%);z-index:9999;'
    'background:#ffffff;border:1px solid #d1d5db;border-radius:6px;'
    'padding:6px 12px;display:flex;align-items:center;gap:8px;'
    'box-shadow:0 2px 6px rgba(0,0,0,.15);font-size:12px;">'
    '<span style="font-weight:700;white-space:nowrap;">Scenario:</span>'
    '<div style="display:flex;gap:4px;">' + _tabs + '</div>'
    '</div>'
)
m.get_root().html.add_child(Element(scenario_ctrl_html))

# ------------------------------------------------------------
# MAP TOOLS
# ------------------------------------------------------------
plugins.Fullscreen().add_to(m)
plugins.MeasureControl(position='topleft', primary_length_unit='meters').add_to(m)
plugins.MousePosition(position='topright').add_to(m)
plugins.Draw(
    export=True,
    filename='candidate_area.geojson',
    position='topleft',
    draw_options={
        'polyline': False,
        'rectangle': True,
        'circle': False,
        'marker': False,
        'circlemarker': False,
    },
    edit_options={'edit': True, 'remove': True},
).add_to(m)

# ------------------------------------------------------------
# LEGEND + GUIDANCE
# ------------------------------------------------------------
legend_html = (
    '<div style="position: fixed; bottom: 18px; left: 18px; z-index: 9999; background: white;'
    ' border: 1px solid #d1d5db; border-radius: 6px; padding: 10px 12px;'
    ' font-size: 12px; max-width: 350px;">'
    '<div style="font-weight: 700; margin-bottom: 6px;">Housing Suitability POC</div>'
    '<div style="margin-bottom: 6px;">Use <b>Scenario</b> tabs (top) to switch between suitability scenarios.</div>'
    '<div style="margin-bottom: 6px;">Use <b>Layer Control</b> to toggle scenario raster masks and constraint overlays.</div>'
    '<div style="margin-bottom: 6px;">Building popup: ID, display score, and top 3 worst factors.</div>'
    '<div style="margin-bottom: 6px;">Draw a polygon/rectangle to mark a candidate area and export it as GeoJSON.</div>'
    '<div><b>Suitability Classes (Non-Constraint)</b></div>'
    + ''.join(
        f'<div><span style="background:{SUIT_COLORS[i]};">&nbsp;&nbsp;&nbsp;</span> {SUIT_LABELS[i]}</div>'
        for i in range(5)
    ) +
    '<hr style="margin: 6px 0;">'
    '<div><span style="background:#dc2626;">&nbsp;&nbsp;&nbsp;</span> Constraint area</div>'
    '<div><span style="background:#7f1a1a;">&nbsp;&nbsp;&nbsp;</span> Buildings in constraint</div>'
    '</div>'
)
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl(collapsed=False).add_to(m)

# ------------------------------------------------------------
# SAVE OUTPUTS
# ------------------------------------------------------------
output_dir = os.path.join(OUTPUT_ROOT, 'outputs')
os.makedirs(output_dir, exist_ok=True)
output_map = os.path.join(output_dir, 'field_map_deep_poc.html')
m.save(output_map)

print(f'\nMap saved: {output_map}')
print('Tip: publish this HTML for online proof-of-concept sharing.')
display(IFrame(output_map, width='100%', height='680'))


FINAL MAP
Unique values in constraint raster: [0 1]
AOI: Mzinyati Stream
Total buildings: 22,196
Constraint buildings: 4,268
Non-constraint buildings: 17,928
Constraint polygons: 129

✅ Map saved: /content/drive/MyDrive/Durban/outputs/outputs/final_map.html
